# 动手实验室 L3: Clustering & Monte Carlo Simulation
---
*实证经济学分析 (2026Spring)*

### 实验室目标
1. **复现 BDM (2004) 的 45% 过度拒绝结论**：通过 Monte Carlo 模拟，亲眼见证不聚类时，由于时间序列相关性导致的严重过度拒绝（假阳性率高达 ~40-45%）。
2. **对比推断方法**：对比 `iid标准误` / `稳健标准误 (Huber-White)` / `单聚类标准误 (One-way Cluster)` / `双向聚类标准误 (Two-way Cluster)` / `Wild Cluster Bootstrap`。
3. **AR(1) 系数退化速度测试**：改变 AR(1) 自相关系数 $\rho \in \{0, 0.4, 0.8\}$，观察推断方法的退化速度。

## 1. 理论背景：为什么不聚类会发生 45% 的过度拒绝？

在双重差分 (DID) 设定下，政策一般是在较粗的层级（如省份级 `province`）实施的，而我们的观测数据通常在更细的层级（如企业或个体-年份 `i, t`）。
**Bertrand, Duflo, and Mullainathan (2004, QJE)** 指出，如果我们忽略误差项在组内的序列相关性：
- 传统的 OLS 标准误会严重低估真实标准误。
- 在真实政策效应为 0 时，检验的假阳性率（拒绝率）名义上是 5%，实际会暴增到 **35% - 45%**。

**解决黄金准则 (Cameron & Miller, 2015)**：
1. 必须在**政策分配的层级**进行聚类（例如省份级聚类）。
2. 如果聚类数 $G < 50$（例如只有 30 个省份），渐近分布失效，应使用 **Wild Cluster Bootstrap** 或 **Randomization Inference**。

## 2. Stata 模拟代码：复现 BDM (2004)

以下是在 Stata 环境下运行 Monte Carlo 模拟的典型代码。我们通过随机分配虚拟政策来复现过高拒绝率。

```stata
* ========================================== *
* Stata 模拟: BDM(2004) 45% 过度拒绝复现
* ========================================== *

clear all
set obs 500         // 500 个个体
gen id = _n
expand 10           // 10 个时期
bysort id: gen t = _n

* 构造个体固定效应和时间趋势
gen u_i = rnormal(0, 1)
gen v_t = rnormal(0, 1)

* 构造 AR(1) 误差项 (自相关系数 rho = 0.8)
gen eps = .
bysort id: replace eps = rnormal(0, 1) if t == 1
bysort id: replace eps = 0.8 * eps[_n-1] + rnormal(0, 1) if t > 1

* 生成虚拟解释变量和控制变量
gen x = rnormal(0, 1)
gen g = ceil(id / 10)  // 将500个个体分为50个组 (类似省份)

save "simdata.dta", replace

* 进行 Monte Carlo 模拟循环
global N = 300
matrix Result = J($N, 4, .)

set seed 12345
forv i = 1/$N {
    use "simdata.dta", clear
    
    * 随机分配 10 个组作为处理组，政策在 t >= 5 发生
    cap drop grnum gt year treat
    qui g grnum = .
    forv j = 1/50 {
        scalar r = runiform(0,1)
        qui replace grnum = r if g == `j'
    }
    egen gt = cut(grnum), group(5)
    qui gen treated = (gt == 0) // 前 20% 的组作为处理组
    qui gen year = (t >= 5)
    qui gen treat = treated * year
    
    * 真实效应为 0
    qui gen y = u_i + v_t + 0 * treat + eps
    
    * 1. OLS (无聚类)
    qui reg y treat x i.g i.t
    scalar p_iid = 2 * ttail(e(df_r), abs(_b[treat]/_se[treat]))
    
    * 2. 稳健标准误 (HC1)
    qui reg y treat x i.g i.t, r
    scalar p_hc = 2 * ttail(e(df_r), abs(_b[treat]/_se[treat]))
    
    * 3. 组内聚类标准误 (按省份级 g 聚类)
    qui reg y treat x i.g i.t, vce(cluster g)
    scalar p_cl = 2 * ttail(e(N_clust) - 1, abs(_b[treat]/_se[treat]))
    
    * 4. 稳健面板估计 (xtreg, fe)
    xtset id t
    qui xtreg y treat x i.t, fe vce(cluster g)
    scalar p_fe_cl = 2 * ttail(e(N_clust) - 1, abs(_b[treat]/_se[treat]))
    
    matrix Result[`i', 1] = (p_iid, p_hc, p_cl, p_fe_cl)
}

* 计算假阳性率 (名义显著性水平 5% 时的实际拒绝率)
svmat Result, names(p)
gen rej_iid   = (p1 < 0.05)
gen rej_hc    = (p2 < 0.05)
gen rej_cl    = (p3 < 0.05)
gen rej_fe_cl = (p4 < 0.05)

tabstat rej_iid rej_hc rej_cl rej_fe_cl, stat(mean)
```

## 3. Python 模拟与 AR(1) 退化速度分析

为了更直观地展示在自相关系数 $\rho \in \{0, 0.4, 0.8\}$ 下各种标准误的退化表现，我们使用 Python 重新运行 Monte Carlo 实验并绘图。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def run_mc_simulation(rho=0.8, n_units=500, n_periods=10, n_sims=200, seed=42):
    rng = np.random.default_rng(seed)
    
    p_iid_list = []
    p_hc_list = []
    p_cl_list = []
    
    for sim in range(n_sims):
        # 生成固定效应
        unit_fe = rng.normal(0, 1, n_units)
        time_fe = rng.normal(0, 1, n_periods)
        
        # 构造 AR(1) 误差
        eps = np.zeros((n_units, n_periods))
        eps[:, 0] = rng.normal(0, 1, n_units)
        for t in range(1, n_periods):
            eps[:, t] = rho * eps[:, t - 1] + rng.normal(0, 1, n_units)
            
        # 随机分配 100 个省份级组 (每个省有 5 个个体)
        n_groups = 50
        group_ids = np.repeat(np.arange(n_groups), n_units // n_groups)
        
        # 随机分配处理组省份
        treated_groups = rng.choice(n_groups, size=n_groups // 5, replace=False)
        is_treated_unit = np.isin(group_ids, treated_groups)
        
        # 政策矩阵 (t >= 5)
        D = np.zeros((n_units, n_periods))
        D[is_treated_unit, 5:] = 1
        
        # 观测值 Y (真实效应为 0)
        Y = unit_fe[:, None] + time_fe[None, :] + 0 * D + eps
        
        # 双向去均值得到面版内估计 (Within Transformation)
        Y_dm = Y - Y.mean(axis=1, keepdims=True) - Y.mean(axis=0) + Y.mean()
        D_dm = D - D.mean(axis=1, keepdims=True) - D.mean(axis=0) + D.mean()
        
        y_flat = Y_dm.flatten()
        x_flat = D_dm.flatten()
        
        # 最小二乘估计
        beta_hat = (x_flat * y_flat).sum() / (x_flat ** 2).sum()
        resid = y_flat - beta_hat * x_flat
        
        df_adj = n_units * n_periods - n_units - n_periods
        
        # 1. iid 标准误
        se_iid = np.sqrt((resid ** 2).sum() / df_adj / (x_flat ** 2).sum())
        t_iid = abs(beta_hat / se_iid)
        p_iid = 2 * (1 - stats.t.cdf(t_iid, df=df_adj))
        
        # 2. HC1 稳健标准误
        se_hc = np.sqrt(((x_flat * resid) ** 2).sum() / (x_flat ** 2).sum() ** 2 * (n_units * n_periods) / df_adj)
        t_hc = abs(beta_hat / se_hc)
        p_hc = 2 * (1 - stats.t.cdf(t_hc, df=df_adj))
        
        # 3. 按省份级 group_ids 聚类
        # 重新排整为 (n_groups, n_per_group * n_periods)
        x_reshaped = x_flat.reshape(n_groups, -1)
        r_reshaped = resid.reshape(n_groups, -1)
        group_scores = (x_reshaped * r_reshaped).sum(axis=1)
        se_cl = np.sqrt((group_scores ** 2).sum() / (x_flat ** 2).sum() ** 2 * n_groups / (n_groups - 1))
        t_cl = abs(beta_hat / se_cl)
        p_cl = 2 * (1 - stats.t.cdf(t_cl, df=n_groups - 1))
        
        p_iid_list.append(p_iid)
        p_hc_list.append(p_hc)
        p_cl_list.append(p_cl)
        
    rej_iid = (np.array(p_iid_list) < 0.05).mean() * 100
    rej_hc = (np.array(p_hc_list) < 0.05).mean() * 100
    rej_cl = (np.array(p_cl_list) < 0.05).mean() * 100
    
    return rej_iid, rej_hc, rej_cl

现在我们计算不同自相关系数 $\rho \in \{0, 0.4, 0.8\}$ 下三种推断方法的假阳性率，观察退化过程。

In [ ]:
rhos = [0.0, 0.4, 0.8]
results = {}

for r in rhos:
    print(f"正在模拟 rho = {r} ...")
    results[r] = run_mc_simulation(rho=r, n_sims=150)

df_res = pd.DataFrame(results, index=['iid 标准误', 'HC1 稳健', '省份级聚类']).T
print("\n实验结果对比 (显著性水平 5%):")
print(df_res)

## 4. 结果可视化：直观感受“45%假阳性”的震撼

In [ ]:
ax = df_res.plot(kind='bar', figsize=(10, 6), color=['#AE0B2A', '#D9A441', '#003153'], edgecolor='black')
plt.axhline(5, color='gray', linestyle='--', linewidth=1.5, label='名义 5% 水平')
plt.title('自相关系数 \u03c1 对名义 5% 假阳性率的影响 (退化速度)', fontsize=14)
plt.xlabel('自相关系数 \u03c1', fontsize=12)
plt.ylabel('实际拒绝率 (%)', fontsize=12)
plt.ylim(0, max(df_res.values.max() + 5, 20))
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.xticks(ticks=[0, 1, 2], labels=['\u03c1 = 0.0 (iid)', '\u03c1 = 0.4 (温和自相关)', '\u03c1 = 0.8 (严重自相关)'], rotation=0)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.5),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

### 5. 课后总结与行动指南

1. **当 $\rho = 0.0$ 时**：残差之间没有序列相关，iid、稳健、聚类标准误的表现均在 5% 水平线附近波动，推断是有效的。
2. **当 $\rho = 0.4$ 时**：存在轻度序列相关。iid 和稳健标准误开始出现显著退化，假阳性率可能达到 15% - 20%。
3. **当 $\rho = 0.8$ 时**：存在高度序列相关（符合绝大多数面板经济数据的典型特征）。iid 和稳健标准误彻底失效，实际拒绝率飙升至 **35% - 45%**，这就是 BDM (2004) 著名的震撼发现！而聚类标准误能够把假阳性率成功拉回 5% 附近，完成完美拯救。

**黄金法则**：*无论做面板回归还是 DID，永远在处理/政策实施的分组层级上聚类（vce(cluster group) / cluster = ~group）！*